# 05 — Results Visualization & Statistical Analysis

**Runtime:** CPU-only. Tidak butuh GPU.

**Input:** `artifacts/logs/evaluation/results_master.csv` + `homography_results.csv`

**Output:** Publication-ready figures (300 DPI) + LaTeX tables + statistical test results.

In [ ]:
!pip install matplotlib seaborn scipy pandas --quiet
print('Dependencies ready')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Environment & Path Setup — kompatibel Kaggle DAN Colab (auto-detect)
# ══════════════════════════════════════════════════════════════════════
import os, sys

def _detect_env():
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    if os.path.exists('/content'):
        return 'colab'
    return 'local'

ENV = _detect_env()
print(f'Environment terdeteksi: {ENV}')

# ⚠️ GANTI dengan URL repo GitHub kamu sendiri (isi SEKALI saja per notebook)
GITHUB_REPO = 'https://github.com/atangorp/soccer-homography-benchmark.git'

def _link(src, dst):
    """Symlink src -> dst kalau src ada dan dst belum ada. Aman dipanggil berkali-kali."""
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(src, dst)

if ENV == 'kaggle':
    PROJECT_ROOT = '/kaggle/working/soccer-homography-benchmark'
    if not os.path.exists(PROJECT_ROOT):
        os.system(f'git clone -q {GITHUB_REPO} "{PROJECT_ROOT}"')
    sys.path.insert(0, PROJECT_ROOT)

    # WAJIB: attach dataset "soccer-field-raw" via "+ Add Input" di sidebar kanan
    # (ganti 'soccer-field-raw' kalau slug dataset kamu berbeda)
    _link('/kaggle/input/soccer-field-raw',
          f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8')
    # Output evaluasi — attach "03-evaluation" DAN "04-homography-pipeline" via "+ Add Input"
    _link('/kaggle/input/03-evaluation/soccer-homography-benchmark/artifacts/logs/evaluation/results_master.csv',
          f'{PROJECT_ROOT}/artifacts/logs/evaluation/results_master.csv')
    _link('/kaggle/input/03-evaluation/soccer-homography-benchmark/artifacts/logs/evaluation/results_per_image.csv',
          f'{PROJECT_ROOT}/artifacts/logs/evaluation/results_per_image.csv')
    _link('/kaggle/input/04-homography-pipeline/soccer-homography-benchmark/artifacts/logs/evaluation/homography_results.csv',
          f'{PROJECT_ROOT}/artifacts/logs/evaluation/homography_results.csv')

elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/soccer-homography-benchmark'
    sys.path.insert(0, PROJECT_ROOT)

else:
    PROJECT_ROOT = os.getcwd()
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

FIG_DIR = f'{PROJECT_ROOT}/artifacts/results/figures'
TBL_DIR = f'{PROJECT_ROOT}/artifacts/results/tables'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TBL_DIR, exist_ok=True)

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 10, 'axes.titlesize': 11,
    'axes.labelsize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})
print('Setup complete')
print(f'PROJECT_ROOT : {PROJECT_ROOT}')


In [ ]:
# ── Cell 2: Load results ─────────────────────────────────────────────────────
df_master = pd.read_csv(f'{PROJECT_ROOT}/artifacts/logs/evaluation/results_master.csv')
df_homo   = pd.read_csv(f'{PROJECT_ROOT}/artifacts/logs/evaluation/homography_results.csv')

print(f'Master CSV : {len(df_master)} rows, columns: {list(df_master.columns)}')
print(f'Homography : {len(df_homo)} rows')
df_master.head()

In [ ]:
# ── Cell 3: Figure 1 — MPE comparison bar chart ──────────────────────────────
FAMILY_COLORS = {
    'yolo11'  : '#2563EB',
    'hrnet'   : '#16A34A',
    'vitpose' : '#7C3AED',
    'detr'    : '#DC2626',
}

fig, ax = plt.subplots(figsize=(10, 4.5))

df_plot = df_master.sort_values('mpe_mean')
colors  = [FAMILY_COLORS.get(f, '#888') for f in df_plot['model_family']]

bars = ax.bar(
    df_plot['model_full_name'], df_plot['mpe_mean'],
    yerr=df_plot['mpe_std'], capsize=4,
    color=colors, alpha=0.85, edgecolor='white', linewidth=0.5,
)

ax.set_xlabel('Model')
ax.set_ylabel('Mean Pixel Error (px)')
ax.set_title('Keypoint Detection Accuracy — Mean Pixel Error (lower is better)')
ax.tick_params(axis='x', rotation=40)
ax.grid(axis='y', alpha=0.3, linewidth=0.5)
ax.spines[['top', 'right']].set_visible(False)

# Legend per family
for family, color in FAMILY_COLORS.items():
    ax.bar(0, 0, color=color, label=family.upper())
ax.legend(title='Architecture family', loc='upper right', framealpha=0.9)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_mpe_comparison.pdf')
plt.savefig(f'{FIG_DIR}/fig_mpe_comparison.png')
plt.show()
print(f'✅ Saved: fig_mpe_comparison')

In [ ]:
# ── Cell 4: Figure 2 — Accuracy vs Efficiency scatter ────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

for _, row in df_master.iterrows():
    color = FAMILY_COLORS.get(row['model_family'], '#888')
    ax.scatter(row['fps'], row['mpe_mean'], s=row['model_size_mb']*0.8 + 40,
               c=color, alpha=0.85, edgecolors='white', linewidths=0.8)
    ax.annotate(row['model_full_name'].replace('Deformable-DETR-', 'DETR-'),
                (row['fps'], row['mpe_mean']),
                xytext=(5, 5), textcoords='offset points', fontsize=7.5)

ax.set_xlabel('Inference Speed (FPS) →  faster')
ax.set_ylabel('Mean Pixel Error (px) →  more accurate')
ax.set_title('Accuracy vs Efficiency Trade-off\n(bubble size = model file size)')
ax.invert_yaxis()  # Lower MPE = better = top of chart
ax.grid(alpha=0.25, linewidth=0.5)
ax.spines[['top', 'right']].set_visible(False)

for family, color in FAMILY_COLORS.items():
    ax.scatter([], [], c=color, label=family.upper(), s=60)
ax.legend(title='Family', framealpha=0.9)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_accuracy_efficiency_tradeoff.pdf')
plt.savefig(f'{FIG_DIR}/fig_accuracy_efficiency_tradeoff.png')
plt.show()

In [ ]:
# ── Cell 5: Figure 3 — Homography validity rate ──────────────────────────────
homo_summary = df_homo.groupby(['model_family', 'model_variant']).agg(
    validity_rate=('is_H_valid', 'mean'),
    model_full_name=('model_variant', lambda x: x.iloc[0]),
).reset_index()

# Buat full name
homo_summary['model_full_name'] = (
    homo_summary['model_family'].str.upper() + '-' +
    homo_summary['model_variant'].str.upper()
)
homo_summary = homo_summary.sort_values('validity_rate', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Subplot 1: Overall validity rate
colors = [FAMILY_COLORS.get(f, '#888') for f in homo_summary['model_family']]
axes[0].barh(homo_summary['model_full_name'], homo_summary['validity_rate']*100,
             color=colors, alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Homography Validity Rate (%)')
axes[0].set_title('Overall Homography Validity Rate')
axes[0].axvline(x=80, color='gray', linestyle='--', alpha=0.5, label='80% threshold')
axes[0].grid(axis='x', alpha=0.3)
axes[0].spines[['top', 'right']].set_visible(False)

# Subplot 2: Broadcast vs Scouting
for src, style in [('broadcast', 'solid'), ('scouting', 'dashed')]:
    subset = df_homo[df_homo['image_source'] == src]
    if len(subset) == 0:
        continue
    rates = subset.groupby(['model_family','model_variant'])['is_H_valid'].mean()
    axes[1].plot(range(len(rates)), rates.values*100,
                 marker='o', linestyle=style, label=src.capitalize())

axes[1].set_xticks(range(len(rates)))
axes[1].set_xticklabels([
    f"{f}-{v}" for f,v in rates.index
], rotation=40, ha='right', fontsize=8)
axes[1].set_ylabel('Validity Rate (%)')
axes[1].set_title('Broadcast vs Scouting Data')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_homography_validity.pdf')
plt.savefig(f'{FIG_DIR}/fig_homography_validity.png')
plt.show()

In [ ]:
# ── Cell 6: Statistical significance tests ───────────────────────────────────
from src.evaluation.statistical_tests import (
    kruskal_wallis_test, pairwise_wilcoxon, check_normality
)

# Load per-image results untuk statistical testing
df_per_image = pd.read_csv(f'{PROJECT_ROOT}/artifacts/logs/evaluation/results_per_image.csv')

# Build dict: model_name → array of MPE values
errors_dict = {}
for model_name, grp in df_per_image.groupby('model_full_name'):
    errors_dict[model_name] = grp['mpe'].dropna().values

# Kruskal-Wallis: apakah ada perbedaan signifikan?
print('\n── Kruskal-Wallis Test (semua model) ──')
kw = kruskal_wallis_test(errors_dict)
print(kw['interpretation'])

# Pairwise Wilcoxon (jika KW significant)
if kw['significant']:
    print('\n── Pairwise Wilcoxon Test ──')
    df_wilcoxon = pairwise_wilcoxon(errors_dict)
    df_sig = df_wilcoxon[df_wilcoxon['significant']].sort_values('p_value')
    print(f'Significant pairs: {len(df_sig)}/{len(df_wilcoxon)}')
    display(df_sig[['model_a','model_b','p_value','effect_size_d','interpretation']].head(15))
    
    # Save
    df_wilcoxon.to_csv(f'{TBL_DIR}/statistical_tests.csv', index=False)
    print(f'✅ Saved: statistical_tests.csv')

In [ ]:
# ── Cell 7: Generate LaTeX results table ─────────────────────────────────────

def generate_latex_table(df: pd.DataFrame) -> str:
    cols = ['rank','model_full_name','mpe_mean','mpe_std','fps','model_size_mb','detection_rate']
    df_tbl = df[cols].copy()
    df_tbl.columns = ['Rank','Model','MPE (px)','±Std','FPS','Size (MB)','Det. Rate']
    
    latex = df_tbl.to_latex(
        index=False,
        float_format='%.2f',
        caption='Performance comparison of 11 pose estimation architectures for soccer field '
                'keypoint detection and homography prediction.',
        label='tab:results_comparison',
        column_format='c l c c c c c',
        escape=False,
    )
    return latex

latex_str = generate_latex_table(df_master)
tex_path  = f'{TBL_DIR}/results_table.tex'
with open(tex_path, 'w') as f:
    f.write(latex_str)

print(f'✅ LaTeX table saved: {tex_path}')
print('\nPreview:')
print(latex_str[:600])

In [ ]:
# ── Cell 8: Print all figure paths ───────────────────────────────────────────
import glob
figs = glob.glob(f'{FIG_DIR}/*.png') + glob.glob(f'{FIG_DIR}/*.pdf')
print(f'\n📁 Generated {len(figs)} files:')
for f in sorted(figs):
    size_kb = os.path.getsize(f) / 1024
    print(f'  {os.path.basename(f):<50} ({size_kb:.0f} KB)')